# GNSS Interferometric Reflectometry (GNSS-IR)

**Version:** 1.0 | **Last updated:** 2026-08-18 

**Author:** Henry Berglund, Alex Hamilton, Eshanta Mishra | **Author institution:** EarthScope Consortium

**Maintainer:** EarthScope OnRamp Team | **Maintainer's contact :** help@earthscope.org

**Estimated Time:**  ~ 45 minutes | **Pathway:** MVP1

**License:** CC-BY-4.0

## Introduction

**What this notebook does:** It retrieves Signal-to-Noise ratio (SNR) observations and satellite elevation/azimuth angles for a single GNSS station using the EarthScope SDK, then recovers the height of the antenna above the surrounding ground surface from the interference pattern hidden in the SNR record.

**Why it is useful:** A GNSS antenna is normally treated as a positioning instrument, and the ground-reflected signal that contaminates it is treated as noise to be suppressed. GNSS-IR turns that noise into a measurement. The GNSS stations become environmental sensors, recording snow depth, soil moisture, and sea level from the reflecting surface beneath them, with no extra hardware.

**What you will accomplish:** By the end of this notebook, you will have fetched SNR and satellite geometry for one station-day, joined them, isolated individual rising and setting satellite arcs, detrended each arc to expose the reflection fringes, and converted the dominant fringe frequency into a reflector height using a Lomb-Scargle periodogram.

---

### Prerequisites

Before starting this notebook, you should:

* [ ] Have completed: [Notebook 1 - Accessing GNSS Observations with the EarthScope SDK](https://github.com/EarthScope/geolab-tutorials/blob/main/geodetic/01-access-gnss-via-SDK.ipynb).
* [ ] Be familiar with basic Python and NumPy arrays.

---

### GeoLab Compute Resources

| Setting | Recommended |
|---|---|
| **Image** | GeoLab (default image) |
| **Server size** | 4 GB RAM, ~0.5 CPUs (default server) |

## Learning Objectives

By the end of this notebook, you will be able to:

1. Explain how interference between the direct and ground-reflected GNSS signals encodes antenna height in the SNR record.
2. Retrieve SNR observations and satellite elevation/azimuth from the EarthScope SDK and join them into a single table.
3. Split a satellite track into rising and setting arcs and detrend it.
4. Compute a Lomb-Scargle periodogram for each arc and convert its peak frequency into a reflector height.

## Relevant Documentation & Resources

* [EarthScope SDK documentation](https://docs.earthscope.org/sdk)
* [SciPy `lombscargle` reference](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.lombscargle.html)

## Contents

1. [What is GNSS-IR?](#id-1-what-is-gnss-ir)
2. [Setup & Imports](#id-2-setup-imports)
3. [Retrieve SNR and Satellite Geometry](#id-3-retrieve-snr-and-satellite-geometry)
4. [Merge the Two Streams](#id-4-merge-the-two-streams)
5. [Split into Arcs and Detrend SNR](#id-5-split-into-arcs-and-detrend-snr)
6. [Lomb-Scargle and Reflector Height](#id-6-lomb-scargle-and-reflector-height)
7. [Visualize the Results](#id-7-visualize-the-results)
8. [Exploration Exercises](#id-8-exploration-exercises)
9. [Troubleshooting & Support](#id-9-troubleshooting-support)

## 1. What is GNSS-IR?

A GNSS antenna does not only receive the signal that travels straight down from the satellite. It also receives a copy of that signal that has bounced off the ground. The two copies arrive along slightly different path lengths, so they interfere. That interference is what GNSS-IR measures.

The extra distance travelled by the reflected ray depends on the height $H$ of the antenna phase center above the reflecting surface and on the satellite elevation angle $e$:

$$\delta = 2 H \sin(e)$$

As a satellite rises or sets, $e$ changes continuously, so $\delta$ sweeps through many wavelengths and the two rays cycle in and out of phase. The recorded SNR therefore carries an oscillating pattern. That oscillation is the signal we want.

Because $\delta$ is linear in $\sin(e)$ and not in $e$ itself, the oscillation is only strictly periodic when plotted against $\sin(e)$. This is why every step below uses $\sin(\text{elevation})$ as the x-axis rather than elevation. Once in that coordinate, the fringe frequency $f_{\text{peak}}$ (in cycles per unit of $\sin e$) gives the reflector height directly:

$$H = \frac{\lambda \, f_{\text{peak}}}{2}$$

where $\lambda$ is the carrier wavelength, 0.1903 m for the GPS L1 signal used here.

### Why the 5 to 30 degree elevation window?

* **Below ~5 degrees** the signal is noisy and often blocked by terrain or nearby structures.
* **Between 5 and 30 degrees** the reflected ray strikes the surface at a grazing angle, reflects efficiently, and produces clean, high-amplitude fringes. This is the GNSS-IR working window.
* **Above ~30 degrees** the antenna is designed to reject signals arriving from below, so the reflected component is strongly attenuated and the fringes fade out.

### Why does this matter?

The reflector height is a physical distance to a surface, so anything that moves that surface shows up as a change in $H$:

* **Snow depth** - fresh snow raises the reflecting surface, so $H$ decreases through the winter.
* **Soil moisture** - wet soil reflects more strongly, changing the fringe amplitude rather than the height.
* **Sea and lake level** - the water surface is the reflector, and $H$ tracks the tide.

### The sensing footprint

The reflection does not come from directly beneath the antenna. It comes from an elliptical patch tens of metres away, in the direction of the satellite. This is why the **azimuth** of each arc matters and why we plot the results on a polar diagram in Section 7: different azimuths sample different ground.

## 2. Setup & Imports

In [ ]:
# Standard library imports
import datetime as dt

# Third-party imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import lombscargle

from earthscope_sdk import AsyncEarthScopeClient
from earthscope_sdk.client.data_access.models import GeodeticCoordinate, FloatFilter

es = AsyncEarthScopeClient()

As in Notebook 1, the client finds your EarthScope credentials automatically inside GeoLab. If you run this notebook elsewhere, authenticate once with the [EarthScope CLI](https://gitlab.com/earthscope/public/earthscope-cli) first.

### Configuration

Set your parameters here before running the rest of the notebook. Every later cell reads from these variables, so this is the only place you need to edit.

In [ ]:
# Modify these values before running the notebook.

STATION = "P04000USA"                  # IGS 9-character station ID 

# The station's position, needed to compute satellite elevation and azimuth (see note below).
REF_PT = GeodeticCoordinate(
    latitude=38.0714982807,
    longitude=-102.6869823081,
    height=1102.64837,
    label=STATION,
)

# --- Data retrieval settings ---
DATE = dt.date(2024, 1, 1)             # UTC day to analyze
START = dt.datetime(DATE.year, DATE.month, DATE.day, tzinfo=dt.timezone.utc)
END = START + dt.timedelta(days=1)

SYSTEM = "G"                           # G=GPS, R=GLONASS, E=Galileo, C=BeiDou
OBS_CODE = "1C"                        # RINEX observation code; 1C = GPS L1 C/A
SAMPLE_INTERVAL = dt.timedelta(seconds=15)

# --- GNSS-IR analysis settings ---
ELEV_MIN = 5.0                         # degrees - below this, too noisy / terrain-blocked
ELEV_MAX = 30.0                        # degrees - above this, the reflection is too weak
POLY_DEG = 2                           # polynomial order used to detrend SNR

WAVELENGTH = 0.1903                    # metres, must match OBS_CODE
                                       # GPS L1 = 0.1903, L2 = 0.2442, L5 = 0.2548

# Reflector heights to search over, in metres
H_MIN, H_MAX, H_STEP = 0.1, 10.0, 0.01

print(f"Station {STATION} on {DATE}, signal {SYSTEM}{OBS_CODE} (lambda = {WAVELENGTH} m)")

> **Note on `REF_PT`:** the EarthScope API stores satellite orbits, not viewing angles. To turn an orbit into the elevation and azimuth *as seen from your antenna*, the server needs to know where your antenna is, which is what `REF_PT` supplies. The coordinates above are P040's IGS14 position.
> 
> If you swap in a different station you will need its coordinates too. They come from the GAGE IGS14 coordinate catalogue, one line per station in the form `4CHARID, name, latitude, longitude, height, ...`.

## 3. Retrieve SNR and Satellite Geometry

**What:** two separate requests to the EarthScope API, each returning an Apache Arrow table for the same station, day, and 15-second sampling.

**Why two requests:** they answer different questions. The **observations** endpoint returns what the receiver actually recorded, including `snr`, the quantity carrying the interference fringes. The **ephemeris positions** endpoint returns where each satellite was in the sky, computed from the broadcast orbits and your `REF_PT`. GNSS-IR needs both: the fringe pattern *and* the elevation angle it is a function of.

We keep only the `snr` field and a single observation code, since a full observation record (pseudorange, carrier phase, flags, every signal) would be far larger and entirely unused here.

**Expected result:** a few tens of thousands of rows in each table for a single day at 15-second sampling.

In [ ]:
# Fetch SNR observations for the chosen signal only.
obs_table = await es.data.gnss_observations(
    start_datetime=START,
    end_datetime=END,
    station_name=STATION,
    system=[SYSTEM],
    obs_code=OBS_CODE,
    field="snr",
    sample_interval=SAMPLE_INTERVAL,
    session_name="A",
).fetch()

obs_df = obs_table.to_pandas()
print(f"{len(obs_df):,} SNR rows - columns: {obs_df.columns.tolist()}")
obs_df.head()

The ephemeris request uses `elevation_filter` to discard satellites outside our working window before the data is ever sent. The two-degree margin is used so that the request does not cut exactly at the boundary the analysis will later use.

In [ ]:
# Fetch satellite elevation and azimuth as seen from the station.
eph_table = await es.data.gnss_ephemeris_positions(
    start_datetime=START,
    end_datetime=END,
    system=[SYSTEM],
    field=["elevation", "azimuth"],
    reference_point=REF_PT,
    elevation_filter=FloatFilter(min=ELEV_MIN - 2),
    sample_interval=SAMPLE_INTERVAL,
).fetch()

eph_df = eph_table.to_pandas()
print(f"{len(eph_df):,} ephemeris rows - columns: {eph_df.columns.tolist()}")
eph_df.head()

### The fields we will use

| Column | Source | Meaning |
|---|---|---|
| `timestamp` | both | Epoch of the measurement, in UTC. |
| `satellite` | both | Satellite number within its constellation (its PRN). |
| `system` | both | Constellation code; `G` for GPS here. |
| `obs_code` | observations | Which signal was measured, e.g. `1C` for GPS L1 C/A. |
| `snr` | observations | Signal-to-noise ratio in dB-Hz. This is the quantity carrying the fringes. |
| `igs` | observations | The station's IGS 9-character name. |
| `elevation` | ephemeris | Satellite elevation angle above the horizon, in degrees. |
| `azimuth` | ephemeris | Satellite azimuth, in degrees clockwise from north. |
| `label` | ephemeris | Echoes the `label` you set on `REF_PT`, i.e. the station name. |

## 4. Merge the Two Streams

Each SNR measurement must be paired with the elevation and azimuth of the *same satellite* at the *same instant*. Because both requests used the same `sample_interval`, their timestamps line up exactly and a plain inner join is enough.

The ephemeris table names the station `label` while the observation table names it `igs`, so we rename one before joining. Including the station in the join key is deliberate: if you later extend this notebook to several stations at once, it prevents one station's SNR from being paired with another station's viewing geometry.

In [ ]:
merged = (
    obs_df.merge(
        eph_df.rename(columns={"label": "igs"}),
        on=["timestamp", "system", "satellite", "igs"],
        how="inner",
    )
    .sort_values(["timestamp", "system", "satellite"])
    .reset_index(drop=True)
)

print(f"Merged rows: {len(merged):,}")
print(f"Satellites:  {sorted(merged['satellite'].unique().tolist())}")
merged.head()

> **Check:** the merged table should be somewhat smaller than the SNR table, because epochs where the satellite was below the elevation filter drop out. You should see roughly 30 GPS satellites. If the merge returns zero rows, one of the two requests came back empty, or the station has no data on this date.

## 5. Split into Arcs and Detrend SNR

### What is an arc?

An **arc** is one continuous pass of a single satellite through our elevation window: either a rise or a set, but not both. Arcs are the natural unit of GNSS-IR because within a single arc the elevation changes monotonically, so the fringes form one clean, uninterrupted oscillation. Mixing a rise and the following set into a single record would fold two different reflection footprints together and smear the periodogram.

We break a satellite's record into arcs at two kinds of boundary:

* **Time gaps** longer than 10 minutes, which indicate the satellite dropped out and came back.
* **Elevation reversals**, where the satellite stops rising and begins setting. This is detected as a sign change between consecutive elevation differences.

Arcs shorter than 10 points are discarded as too short to yield a meaningful spectrum.

In [ ]:
def split_arcs(df, gap_threshold_min=10):
    """Split a single-satellite DataFrame into rising/setting arcs."""
    df = df.sort_values("timestamp").reset_index(drop=True)

    time_diff = df["timestamp"].diff().dt.total_seconds().fillna(0)
    time_gap = time_diff > gap_threshold_min * 60

    elev_diff = df["elevation"].diff()
    # A sign change between consecutive elevation differences marks the arc apex
    elev_rev = ((elev_diff * elev_diff.shift(1)) < 0).fillna(False)

    arc_id = (time_gap | elev_rev).cumsum()
    return [g.reset_index(drop=True) for _, g in df.groupby(arc_id) if len(g) >= 10]

### Why detrend?

The raw SNR of an arc is dominated by the **direct** signal, which grows smoothly as the satellite climbs and the antenna gain increases. That smooth rise is a hundred times larger than the reflection fringes sitting on top of it, and it would completely dominate any spectrum we computed.

Detrending removes it. We fit a low-order polynomial (degree 2 by default) in $\sin(\text{elevation})$ and subtract it. The polynomial is far too smooth to absorb the rapid fringes, so what remains, `snr_detrend`, is the reflected component alone, oscillating around zero.

Two guards reject arcs that cannot be detrended reliably: too few valid SNR points, or an elevation span under 2 degrees (too short to contain even one full fringe).

In [ ]:
def detrend_snr(arc, poly_deg=POLY_DEG):
    """Remove a polynomial trend from SNR using sin(elevation) as x.

    Returns the arc with 'snr_detrend' and 'sin_elev' columns added,
    or None if the arc is unusable.
    """
    x = np.sin(np.radians(arc["elevation"].values))
    y = arc["snr"].values

    valid = np.isfinite(y)
    if valid.sum() < poly_deg + 2:
        return None
    if arc["elevation"].max() - arc["elevation"].min() < 2.0:
        return None  # too short an arc to detrend reliably

    coeffs = np.polyfit(x[valid], y[valid], poly_deg)
    trend = np.polyval(coeffs, x)

    arc = arc.copy()
    arc["snr_detrend"] = y - trend
    arc["sin_elev"] = x
    return arc

Now apply the elevation mask and build every usable arc for the day.

In [ ]:
filtered = merged[
    (merged["obs_code"] == OBS_CODE)
    & merged["snr"].notna()
    & (merged["elevation"] >= ELEV_MIN)
    & (merged["elevation"] <= ELEV_MAX)
].copy()

all_arcs = []
for sat, grp in filtered.groupby("satellite"):
    for arc in split_arcs(grp):
        arc = detrend_snr(arc)
        if arc is not None:
            arc["satellite"] = sat
            all_arcs.append(arc)

print(f"Found {len(all_arcs)} arcs across {filtered['satellite'].nunique()} satellites")

> **Check:** you should get roughly 60 to 100 arcs for a single GPS day across about 30 satellites. Each satellite passes overhead once or twice and contributes a rising and a setting arc each time. If you get zero, widen the elevation window or confirm that the merge in Section 4 returned data.

### Seeing the fringes

Before running any spectral analysis, look at a single arc directly. This is the whole physical basis of GNSS-IR in one picture: a smooth trend, and a regular oscillation superimposed on it.

We deliberately choose a long arc that stays **below 15 degrees** elevation. As Section 1 explained, the reflected signal weakens as the satellite climbs, so an arc spanning the full 5 to 30 degree window has crisp fringes at its start that fade into noise by its end. A low-elevation arc shows the effect at its clearest.

**What to look for:** the left panel shows raw SNR climbing smoothly with elevation, with a visible ripple along the way. The right panel shows the same arc after the trend is removed, where that ripple becomes a clear oscillation centred on zero. Count its cycles by eye: more cycles across the arc means a higher antenna, since the fringe rate is proportional to $H$.

In [ ]:
# Pick the longest arc that stays at low elevation, where the fringes are clearest
low_arcs = [a for a in all_arcs if a["elevation"].max() <= 15]
example = max(low_arcs or all_arcs, key=len)
sat_id = example["satellite"].iloc[0]

fig, (ax_raw, ax_det) = plt.subplots(1, 2, figsize=(13, 4))

ax_raw.plot(example["sin_elev"], example["snr"], "0.5", lw=1.2)
ax_raw.set_xlabel("sin(elevation)")
ax_raw.set_ylabel("SNR (dB-Hz)")
ax_raw.set_title(f"Raw SNR - satellite {sat_id}")

ax_det.plot(example["sin_elev"], example["snr_detrend"], "steelblue", lw=1.2)
ax_det.axhline(0, color="0.7", lw=0.8)
ax_det.set_xlabel("sin(elevation)")
ax_det.set_ylabel("Detrended SNR (dB-Hz)")
ax_det.set_title(f"Reflection fringes - satellite {sat_id}")

fig.suptitle(f"{STATION}  {DATE}  -  a single satellite arc", fontsize=12)
plt.tight_layout()
plt.show()

## 6. Lomb-Scargle and Reflector Height

Our samples are evenly spaced in *time*, but the x-axis that makes the fringes periodic is $\sin(\text{elevation})$, and elevation does not change at a constant rate. Evenly spaced time therefore becomes unevenly spaced $\sin e$, which other methods of fitting sinusoids such as a standard FFT cannot handle. The Lomb-Scargle periodogram is built for exactly this case: it fits sinusoids by least squares at each trial frequency, with no requirement that the samples be evenly spaced.

### Setting up the frequency grid

We search directly in the quantity we care about. Inverting $H = \lambda f / 2$ from Section 1 gives the frequency corresponding to any candidate height:

$$f = \frac{2H}{\lambda}$$

So we lay out a grid of reflector heights from 0.1 to 10 m, convert each to a frequency, and then to the angular frequency $\omega = 2\pi f$ that SciPy expects. The periodogram's x-axis is then readable as metres of reflector height.

In [ ]:
heights = np.arange(H_MIN, H_MAX, H_STEP)   # candidate reflector heights (m)
freqs = 2 * heights / WAVELENGTH            # cycles per unit of sin(elevation)
omegas = 2 * np.pi * freqs                  # scipy.signal.lombscargle wants angular frequency

print(f"Searching {len(heights):,} heights from {H_MIN} to {H_MAX} m in {H_STEP} m steps")

In [ ]:
def lomb_scargle_arc(arc):
    """Run Lomb-Scargle on a detrended arc.

    Returns (heights, power, peak_height, peak_power) or None if unusable.
    """
    x = arc["sin_elev"].values
    y = arc["snr_detrend"].values

    valid = np.isfinite(x) & np.isfinite(y)
    if valid.sum() < 10:
        return None

    # Remove any residual mean so the power spectrum is meaningful
    y_norm = y[valid] - y[valid].mean()
    pgram = lombscargle(x[valid], y_norm, omegas, normalize=True)

    peak_idx = np.argmax(pgram)
    return heights, pgram, heights[peak_idx], pgram[peak_idx]

Run it over every arc and collect one summary row per arc. Alongside the recovered height we keep the arc's mean azimuth, which tells us which direction the reflection came from, and the peak power, which measures how coherent that reflection was.

In [ ]:
results = []
for arc in all_arcs:
    out = lomb_scargle_arc(arc)
    if out is None:
        continue
    _, _, peak_h, peak_pwr = out
    results.append({
        "satellite": arc["satellite"].iloc[0],
        "azimuth_mean": arc["azimuth"].mean(),
        "elev_mean": arc["elevation"].mean(),
        "t_center": arc["timestamp"].mean(),
        "peak_height": peak_h,
        "peak_power": peak_pwr,
        "n_pts": len(arc),
        "arc": arc,
        "ls": out,
    })

# A tidy summary table, without the bulky arc and periodogram objects
res_df = pd.DataFrame(
    [{k: v for k, v in r.items() if k not in ("arc", "ls")} for r in results]
)

print(f"Processed {len(results)} arcs")
print(res_df.sort_values("peak_power", ascending=False).head(10).to_string(index=False))

> **Check:** for a station on flat, open ground the high-power arcs should cluster around a single reflector height of roughly one to three metres. That number is the physical height of the antenna above the surface. Low-power arcs scatter more widely.

In [ ]:
strong = res_df[res_df["peak_power"] > res_df["peak_power"].median()]
print(f"Median reflector height (all arcs):    {res_df['peak_height'].median():.2f} m")
print(f"Median reflector height (strong arcs): {strong['peak_height'].median():.2f} m")
print(f"Scatter (std) among strong arcs:       {strong['peak_height'].std():.2f} m")

## 7. Visualize the Results

### Arcs and their periodograms

The grid below pairs each of the strongest arcs with its own periodogram, so you can connect the oscillation you see on the left to the peak it produces on the right. The left panels show the detrended SNR only, since the raw-versus-detrended comparison was already made in Section 5.

**What to look for:** a good arc shows a regular, well-formed oscillation and a periodogram with one tall, narrow peak. A poor arc looks irregular and gives a broad or multi-peaked spectrum, which usually means the reflecting surface in that direction is rough, vegetated, or obstructed. Notice that different satellites, sampling different patches of ground, still land on nearly the same height when the surface is uniform.

In [ ]:
top_n = 4
top_results = sorted(results, key=lambda r: r["peak_power"], reverse=True)[:top_n]

fig, axes = plt.subplots(top_n, 2, figsize=(13, top_n * 2.8))
fig.suptitle(f"{STATION}  {DATE}  -  top {top_n} arcs by Lomb-Scargle power", fontsize=13)

for i, r in enumerate(top_results):
    arc = r["arc"]
    hs, pgram, peak_h, peak_pwr = r["ls"]
    ax_snr, ax_ls = axes[i, 0], axes[i, 1]

    # Detrended SNR only; the raw-vs-detrended comparison is in Section 5
    ax_snr.plot(arc["sin_elev"], arc["snr_detrend"], "steelblue", lw=1.2)
    ax_snr.axhline(0, color="0.7", lw=0.8)
    ax_snr.set_xlabel("sin(elevation)")
    ax_snr.set_ylabel("Detrended SNR (dB-Hz)")
    ax_snr.set_title(f"satellite {r['satellite']}  az={r['azimuth_mean']:.0f} deg", fontsize=10)

    ax_ls.plot(hs, pgram, "k", lw=1)
    ax_ls.axvline(peak_h, color="tomato", lw=1.5, ls="--", label=f"H = {peak_h:.2f} m")
    ax_ls.set_xlabel("Reflector height (m)")
    ax_ls.set_ylabel("Normalized power")
    ax_ls.set_title(f"peak = {peak_h:.2f} m   power = {peak_pwr:.3f}", fontsize=10)
    ax_ls.legend(fontsize=8)

plt.tight_layout()
plt.show()

### Reflector height by azimuth

Each arc senses the ground in the direction of its satellite, so plotting height against azimuth is effectively a map of the surface around the station. Radius is reflector height, colour is peak power, and north is at the top.

**What to look for:** a uniform, flat site produces a ring at roughly constant radius all the way around, since every direction sees the same surface at the same distance. A lobe of anomalous heights in one sector points at something real in that direction: sloping ground, a building, or vegetation. Sectors with consistently low power are directions where the reflection is being disrupted.

In [ ]:
fig, ax = plt.subplots(subplot_kw={"projection": "polar"}, figsize=(7, 7))

az_rad = np.radians(res_df["azimuth_mean"].values)
h_vals = res_df["peak_height"].values
pwr = res_df["peak_power"].values

sc = ax.scatter(az_rad, h_vals, c=pwr, s=60,
                cmap="plasma", alpha=0.85, edgecolors="k", lw=0.3)

ax.set_theta_zero_location("N")   # 0 degrees at the top
ax.set_theta_direction(-1)        # azimuth increases clockwise
ax.set_xlabel("Azimuth", labelpad=15)

cbar = plt.colorbar(sc, ax=ax, pad=0.1, shrink=0.7)
cbar.set_label("Lomb-Scargle peak power")
ax.set_title(f"{STATION}  {DATE}\nReflector height (radius) - power (colour)", pad=20)

plt.tight_layout()
plt.show()

## 8. Exploration Exercises

Now that you have completed the core workflow, try modifying the configuration to explore how the results change.

1. **Switch to a different signal:** in the Configuration cell set `OBS_CODE = "2W"` and `WAVELENGTH = 0.2442` (GPS L2), then re-run the notebook. The recovered height should stay about the same. Why? Because the height is a physical property of the site, while the wavelength is like a ruler used to measure it.
   
2. **Change the elevation window:** widen `ELEV_MAX` to 45 degrees, then narrow it to 20 degrees. A wider window gives more points per arc but adds high-elevation data where the reflection is weak. A narrower window gives cleaner fringes but fewer and shorter arcs. Watch both the arc count and the typical peak power.

In [ ]:
# Exploration cell - use this space to experiment

## 9. Troubleshooting & Support

### Common Issues

| Error | Likely cause | Fix |
|---|---|---|
| Either fetch returns 0 rows | The station has no data for that date, or `STATION` is not a valid 9-character IGS ID | Check the station and date in the Configuration cell; try a date you know is covered |
| `merged` is empty but both fetches returned rows | The two requests used different `sample_interval` values, so timestamps do not align | Make sure both fetch cells use the same `SAMPLE_INTERVAL` |
| `Found 0 arcs` | The elevation window is too narrow, or `OBS_CODE` does not match a signal the station records | Widen `ELEV_MIN`/`ELEV_MAX`; check `merged["obs_code"].unique()` for available codes |

### Further Resources

* [EarthScope SDK Documentation](https://docs.earthscope.org/sdk)
* [Authentication using EarthScope CLI](https://gitlab.com/earthscope/public/earthscope-cli) (for non-GeoLab environments)
* [GeoLab Documentation](https://docs.earthscope.org/geolab)
* [GeoLab Community Forum](https://earthscope.discourse.group/latest)